# Entrenamiento en Colab con GPU (basado en `comandos_reducidos_actualizado.txt`)

Este notebook ejecuta solo los comandos **activos** (fuera de bloques entre `"""`).

- `01_matrix_factorization.py`: sin comandos activos (todo marcado como ya ejecutado)
- `02_two_tower.py`: comandos activos incluidos
- `03_two_tower_wide_deep.py`: comandos activos incluidos
- `04_sasrec.py`: comandos activos incluidos


## 1) Configura runtime GPU en Colab
En Colab: `Runtime` -> `Change runtime type` -> `T4/A100/L4 (GPU)`.


In [ ]:
import subprocess
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

nvidia_smi = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True)
if nvidia_smi.returncode == 0:
    print(nvidia_smi.stdout)
else:
    print('nvidia-smi no disponible en este runtime.')

if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        'No hay GPU activa. En Colab ve a Runtime > Change runtime type > Hardware accelerator = GPU, guarda y ejecuta de nuevo.'
    )


## 2) Repo y dependencias
Ajusta `PROJECT_DIR` a la ruta real en tu Colab. Si aÃƒÂºn no has clonado el repo, descomenta la lÃƒÂ­nea de `git clone`.


In [ ]:
from pathlib import Path

# Descomenta y rellena si necesitas clonar en Colab:
!git clone https://github.com/DavidPerez3/pfg-models.git /content/pfg

if Path('/content/pfg/pfg-models').exists():
    PROJECT_DIR = Path('/content/pfg/pfg-models')
elif Path('/content/pfg').exists():
    PROJECT_DIR = Path('/content/pfg')
else:
    raise FileNotFoundError('No existe /content/pfg ni /content/pfg/pfg-models. Clona/monta el repo y vuelve a ejecutar esta celda.')

%cd {PROJECT_DIR}
!pip -q install -r requirements.txt


## 3) (Opcional) Login de W&B


In [ ]:
import os
from getpass import getpass

if not os.environ.get('WANDB_API_KEY'):
    key = getpass('WANDB_API_KEY (Enter para modo offline): ').strip()
    if key:
        os.environ['WANDB_API_KEY'] = key

if os.environ.get('WANDB_API_KEY'):
    os.environ['WANDB_MODE'] = 'online'
    !wandb login --relogin $WANDB_API_KEY
    print('W&B en modo online.')
else:
    os.environ['WANDB_MODE'] = 'offline'
    print('Sin API key: W&B en modo offline (logs locales en el runtime).')


## 4) Helpers y comandos activos
Se aÃ±ade `--device cuda` automÃƒÂ¡ticamente a modelos `02`, `03` y `04` (si no estaba ya en el comando).


In [ ]:
%cd {PROJECT_DIR / 'src' / 'models'}

import subprocess
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device objetivo:', DEVICE)
if DEVICE != 'cuda':
    raise RuntimeError('Este notebook esta preparado para GPU. Activa GPU en Colab antes de entrenar.')

import os
def adapt_command(cmd: str) -> str:
    cmd = cmd.strip()
    if cmd.startswith("python "):
        cmd = cmd.replace("python ", "python -u ", 1)
    if cmd.startswith(("python -u 02_", "python -u 03_", "python -u 04_")) and "--device" not in cmd:
        cmd = f"{cmd} --device {DEVICE}"
    return cmd
def run_cmd(cmd: str, label: str = ""):
    final_cmd = adapt_command(cmd)
    prefix = f"{label} " if label else ""
    print(f"\n{prefix}{final_cmd}")
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    subprocess.run(final_cmd, shell=True, check=True, env=env)
def run_many(commands):
    total = len(commands)
    for i, cmd in enumerate(commands, 1):
        run_cmd(cmd, label=f"[{i}/{total}]")
def run_one(commands, index: int):
    if index < 1 or index > len(commands):
        raise ValueError(f"index fuera de rango: {index}. Rango valido: 1..{len(commands)}")
    run_cmd(commands[index - 1], label=f"[{index}/{len(commands)}]")
def show_commands(commands):
    for i, cmd in enumerate(commands, 1):
        print(f"[{i}] {adapt_command(cmd)}")
CMDS_02 = [
    "python 02_two_tower.py --dataset movielens --dim 128 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group movielens-tt-final",
    "python 02_two_tower.py --dataset amazon_electronics --dim 64 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-tt-final",
    "python 02_two_tower.py --dataset yelp --sample 3000000 --dim 64 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group yelp-tt-validate",
    "python 02_two_tower.py --dataset yelp --sample 3000000 --dim 128 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group yelp-tt-validate",
    "python 02_two_tower.py --dataset yelp --dim 128 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group yelp-tt-final",
    "python 02_two_tower.py --dataset lastfm --sample 3000000 --dim 64 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group lastfm-tt-validate",
    "python 02_two_tower.py --dataset lastfm --sample 3000000 --dim 128 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group lastfm-tt-validate",
    "python 02_two_tower.py --dataset lastfm --dim 128 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group lastfm-tt-final",
    "python 02_two_tower.py --dataset foursquare --sample 3000000 --dim 64 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group foursquare-tt-validate",
    "python 02_two_tower.py --dataset foursquare --sample 3000000 --dim 128 --lr 0.001 --dropout 0.1 --n-neg 2 --batch 1024 --epochs 4 --wandb --wandb-project pfg-recs --wandb-group foursquare-tt-validate",
    "python 02_two_tower.py --dataset foursquare --dim 128 --lr 0.001 --dropout 0.1 --n-neg 5 --batch 1024 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group foursquare-tt-final",
]

CMDS_03 = [
    "python 03_two_tower_wide_deep.py --dataset movielens --sample 1000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group movielens-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset movielens --sample 3000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 5 --wandb --wandb-project pfg-recs --wandb-group movielens-ttwd-final",
    "python 03_two_tower_wide_deep.py --dataset amazon_electronics --sample 1000000 --candidates 500 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset amazon_electronics --sample 1000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset amazon_electronics --sample 3000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 5 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-ttwd-final",
    "python 03_two_tower_wide_deep.py --dataset yelp --sample 1000000 --candidates 500 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group yelp-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset yelp --sample 1000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group yelp-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset yelp --sample 3000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 5 --wandb --wandb-project pfg-recs --wandb-group yelp-ttwd-final",
    "python 03_two_tower_wide_deep.py --dataset foursquare --sample 1000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 2 --wandb --wandb-project pfg-recs --wandb-group foursquare-ttwd-screen",
    "python 03_two_tower_wide_deep.py --dataset foursquare --sample 3000000 --candidates 1000 --lr-wide 0.003 --lr-deep 0.001 --dropout 0.2 --dim 128 --n-neg 5 --wandb --wandb-project pfg-recs --wandb-group foursquare-ttwd-final",
]

CMDS_04 = [
    "python 04_sasrec.py --dataset movielens --max-len 100 --d-model 128 --n-heads 2 --n-layers 2 --dropout 0.1 --neg-samples 5 --lr 0.001 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group movielens-sasrec-final",
    "python 04_sasrec.py --dataset amazon_electronics --max-len 100 --d-model 128 --n-heads 2 --n-layers 2 --dropout 0.1 --neg-samples 5 --lr 0.001 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group amazon_electronics-sasrec-final",
    "python 04_sasrec.py --dataset yelp --max-len 100 --d-model 128 --n-heads 2 --n-layers 2 --dropout 0.1 --neg-samples 5 --lr 0.001 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group yelp-sasrec-final",
    "python 04_sasrec.py --dataset foursquare --max-len 100 --d-model 128 --n-heads 2 --n-layers 2 --dropout 0.1 --neg-samples 5 --lr 0.001 --epochs 10 --wandb --wandb-project pfg-recs --wandb-group foursquare-sasrec-final",
]

print('Comandos cargados:', len(CMDS_02) + len(CMDS_03) + len(CMDS_04))


# Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Detecta raÃƒÂ­z del repo (segÃƒÂºn cÃƒÂ³mo lo hayas clonado)
if Path("/content/pfg/pfg-models").exists():
    REPO_ROOT = Path("/content/pfg/pfg-models")
elif Path("/content/pfg").exists():
    REPO_ROOT = Path("/content/pfg")
else:
    raise FileNotFoundError("No encuentro el repo en /content/pfg ni /content/pfg/pfg-models")

GDRIVE_PROCESSED = Path("/content/drive/MyDrive/5Ã‚Âº/PFG CDIA/processed")
if not GDRIVE_PROCESSED.exists():
    raise FileNotFoundError(f"No existe la ruta en Drive: {GDRIVE_PROCESSED}")

data_dir = REPO_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)

processed_link = data_dir / "processed"
if processed_link.exists():
    print(f"{processed_link} ya existe; no lo modifico.")
else:
    processed_link.symlink_to(GDRIVE_PROCESSED, target_is_directory=True)
    print(f"Enlace creado: {processed_link} -> {GDRIVE_PROCESSED}")

# VerificaciÃƒÂ³n rÃƒÂ¡pida
datasets = ["movielens", "amazon_electronics", "yelp", "lastfm", "foursquare"]
for ds in datasets:
    p = processed_link / ds / "interactions.parquet"
    print(ds, "OK" if p.exists() else "FALTA", p)


## 5) Ejecutar modelo 02 (Two Tower)


In [ ]:
# Modelo 02
show_commands(CMDS_02)
# run_one(CMDS_02, 1)   # Ejecuta solo el comando 1
# run_many(CMDS_02)     # Ejecuta todos los comandos de CMDS_02


## 6) Ejecutar modelo 03 (Two Tower Wide & Deep)


In [ ]:
# Modelo 03
show_commands(CMDS_03)
# run_one(CMDS_03, 1)   # Ejecuta solo el comando 1
# run_many(CMDS_03)     # Ejecuta todos los comandos de CMDS_03


## 7) Ejecutar modelo 04 (SASRec)


In [ ]:
# Modelo 04
show_commands(CMDS_04)
# run_one(CMDS_04, 1)   # Ejecuta solo el comando 1
# run_many(CMDS_04)     # Ejecuta todos los comandos de CMDS_04
